# Part 3: Chunked prefill (Sarathi)

Fill the Part 3 TODOs in `vidur/scheduler/replica_scheduler/sarathi_replica_scheduler.py` first (`pytest tests/test_sarathi_chunk.py`, `pytest tests/test_sarathi_batch.py`). Then run Sarathi at three token budgets from the repo root:

```bash
for C in 256 512 2048; do
  SCHED=sarathi CHUNK=$C QPS=5.53 bash scripts/run_default.sh traces/azure_conversation.csv
done
```

Runs go one after another, about 3 minutes each. If you have plenty of RAM, you can run them in the background instead (add `&` after each command, then `wait`), but each run needs a few GB: four at once ran out of memory on a 16 GB laptop.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "traces").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from analysis.replay import describe_run, load_run
from analysis.workload import load_trace

WAIT_TARGET_S = 2.0    # P99 wait under 2 s
TBT_TARGET_S = 0.100   # P99 time between tokens under 100 ms

trace = load_trace(ROOT / "traces" / "azure_conversation.csv")

# Your Part 2 vLLM run and your three Sarathi runs, any order.
RUN_DIRS = [ROOT / "simulator_output" / d for d in [
    "REPLACE_VLLM", "REPLACE_C256", "REPLACE_C512", "REPLACE_C2048",
]]
runs = {describe_run(d): load_run(d, trace) for d in RUN_DIRS}
print(list(runs))

## Predicted vs measured

Paste your Part 2 Q4 predictions for the P99 time between tokens (in ms). In the table, TBT is the time between tokens and TTFT the time to first token.

In [ ]:
PREDICTED_TBT_P99_MS = {256: None, 512: None, 2048: None}   # from Part 2 Q4

print(f"{'run':16s}{'predicted':>11s}{'P99 TBT':>11s}{'longest':>11s}{'P99 wait':>11s}{'wait > 2 s':>12s}{'median TTFT':>13s}")
for name, run in runs.items():
    tbt = run["tbt"].set_index("percentile").seconds
    wait = run["requests"].wait
    c = int(name.split("C=")[1]) if "C=" in name else None
    pred = PREDICTED_TBT_P99_MS.get(c) if c else None
    pred_s = f"{pred:.0f} ms" if pred else "-"
    print(f"{name:16s}{pred_s:>11s}{tbt[99]*1e3:8.0f} ms{tbt[100]*1e3:8.0f} ms{wait.quantile(.99):9.1f} s{np.mean(wait > WAIT_TARGET_S):12.0%}{run['requests'].ttft.median():11.2f} s")

In [ ]:
fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12, 4))
for name, run in runs.items():
    tbt = run["tbt"]
    ax0.plot(tbt.seconds, tbt.percentile / 100, label=name)
    req = run["requests"]
    ax1.plot(req.arrived_at, req.wait, label=name, linewidth=0.8)
ax0.axvline(TBT_TARGET_S, color="k", linestyle="--", linewidth=0.8)
ax0.set_xscale("log")
ax0.set_xlabel("time between tokens (s)")
ax0.set_ylabel("fraction of tokens")
ax0.legend(frameon=False)
ax1.axhline(WAIT_TARGET_S, color="k", linestyle="--", linewidth=0.8)
ax1.set_xlabel("trace time (s)")
ax1.set_ylabel("wait (s)")
fig.tight_layout()
plt.show()

## Questions

**Q1.** Fill in the table. Which budgets meet the 100 ms target?

| C | Predicted P99 time between tokens (Part 2 Q4) | Measured | Meets 100 ms? |
|---|---|---|---|
| 256 | | | |
| 512 | | | |
| 2048 | | | |

**Your answer:**

**Q2.** Each pass pays about 6 ms fixed plus 78 µs per prompt token, so prompt throughput is C / (6 ms + C × 78 µs). Fill in the table, then pick the budget that meets the 100 ms target with the lowest P99 wait. In one sentence: why does a smaller C cost throughput?

| C | Prompt throughput (tokens/s) | Measured P99 wait | Median time to first token |
|---|---|---|---|
| 256 | | | |
| 512 | | | |
| 2048 | | | |

**Your answer:**

**Q3.** What is the lowest P99 wait across your four runs, and what P99 wait did Part 1 predict for one GPU at full throughput? Can any scheduler meet the 2 s target on one GPU? Answer yes or no, with one sentence on why.

**Your answer:**